# Data Loading & SQL Queries

This notebook connects to the MySQL database `superstore_analytics` and loads the main tables into pandas DataFrames:

- customers
- orders
- products
- order_items

It also runs key SQL queries and stores the results in DataFrames for analysis.

In [ ]:
# --- MySQL Connection Setup ---

from sqlalchemy import create_engine
from urllib.parse import quote_plus
import pandas as pd


username = "root"
password = quote_plus("yourpasswordhere")   
host = "localhost"
database = "superstore_analytics"

# Use the variables here
engine = create_engine(f"mysql+mysqlconnector://{username}:{password}@{host}/{database}")

print("Connection engine created successfully")

Connection engine created successfully


In [2]:
# --- Load tables from MySQL ---
customers = pd.read_sql("SELECT * FROM customers;", engine)
orders = pd.read_sql("SELECT * FROM orders;", engine)
order_items = pd.read_sql("SELECT * FROM order_items;", engine)
products = pd.read_sql("SELECT * FROM products;", engine)

print("Tables loaded successfully")

Tables loaded successfully


In [3]:
# --- Verify tables loaded correctly ---

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Order Items:", order_items.shape)
print("Products:", products.shape)

Customers: (793, 8)
Orders: (5009, 6)
Order Items: (9986, 6)
Products: (1862, 5)


In [4]:
# Query 1: Monthly Revenue Trend
# Purpose: Show total revenue per month and directly load into DataFrame

df1 = pd.read_sql("""
SELECT 
DATE_FORMAT(o.`Order Date`, '%Y-%m') AS `year_month`,
ROUND(SUM(oi.Sales),2) AS revenue
FROM orders o
JOIN order_items oi
ON o.`Order ID` = oi.`Order ID`
GROUP BY `year_month`
ORDER BY `year_month`;
""", engine)

# Display result
df1

,year_month,revenue
0,2014-01,14236.90
1,2014-02,4519.92
2,2014-03,55691.04
3,2014-04,28295.35
4,2014-05,23648.28
5,2014-06,34595.14
6,2014-07,33946.37
7,2014-08,27909.47
8,2014-09,81777.34
9,2014-10,31453.37


In [5]:
# Query 2: Month-over-Month (MoM) Revenue Growth
# Purpose: Calculate monthly revenue growth % and directly load into DataFrame
# First month will show 0% growth instead of NaN

df2 = pd.read_sql("""
WITH monthly_revenue AS
(
SELECT 
DATE_FORMAT(o.`Order Date`, '%Y-%m') AS `year_month`,
SUM(oi.Sales) AS revenue
FROM orders o
JOIN order_items oi
ON o.`Order ID` = oi.`Order ID`
GROUP BY `year_month`
),
                  
revenue_with_lag AS
(
SELECT
`year_month`,
revenue,
LAG(revenue) OVER (ORDER BY `year_month`) AS previous_month_revenue
FROM monthly_revenue
)
                  
SELECT
`year_month`,
revenue,
previous_month_revenue,
ROUND(
(revenue - IFNULL(previous_month_revenue, revenue)) /
IFNULL(previous_month_revenue, revenue) * 100
,2) AS mom_growth_pct
FROM revenue_with_lag
ORDER BY `year_month`;
""", engine)

# Display result
df2

,year_month,revenue,previous_month_revenue,mom_growth_pct
0,2014-01,14236.90,NaN,0.00
1,2014-02,4519.92,14236.90,-68.25
2,2014-03,55691.04,4519.92,1132.12
3,2014-04,28295.35,55691.04,-49.19
4,2014-05,23648.28,28295.35,-16.42
5,2014-06,34595.14,23648.28,46.29
6,2014-07,33946.37,34595.14,-1.88
7,2014-08,27909.47,33946.37,-17.78
8,2014-09,81777.34,27909.47,193.01
9,2014-10,31453.37,81777.34,-61.54


In [6]:
# Query 3: Average Order Value (AOV) & Units Per Transaction
# Purpose: Calculate average revenue per order and average units per transaction per month

df3 = pd.read_sql("""
SELECT 
o.`Customer ID`,
ROUND(SUM(oi.Sales) / COUNT(DISTINCT o.`Order ID`),2) AS avg_order_value_per_customer
FROM orders o
JOIN order_items oi
ON o.`Order ID` = oi.`Order ID`
GROUP BY o.`Customer ID`
ORDER BY avg_order_value_per_customer DESC;
""", engine)

# Display result
df3

,Customer ID,avg_order_value_per_customer
0,SM-20320,5008.61
1,TC-20980,3810.44
2,TA-21385,3648.91
3,GT-14635,3117.07
4,BM-11140,2947.41
...,...,...
788,CJ-11875,16.52
789,SG-20890,15.98
790,RS-19870,11.17
791,LD-16855,5.30


In [7]:
# Query 4: Regional Revenue Performance
# Purpose: Show total revenue by region

df4 = pd.read_sql("""
SELECT 
o.Region,
ROUND(SUM(oi.Sales),2) AS total_revenue
FROM orders o
JOIN order_items oi
ON o.`Order ID` = oi.`Order ID`
GROUP BY o.Region
ORDER BY total_revenue DESC;
""", engine)

# Display result
df4

,Region,total_revenue
0,West,725457.93
1,East,678781.36
2,Central,501239.88
3,South,391721.90


In [8]:
# Query 5: Regional Profitability
# Purpose: Show total profit by region

df5 = pd.read_sql("""
SELECT 
o.Region,
ROUND(SUM(oi.Profit),2) AS total_profit
FROM orders o
JOIN order_items oi
ON o.`Order ID` = oi.`Order ID`
GROUP BY o.Region
ORDER BY total_profit DESC;
""", engine)

# Display result
df5

,Region,total_profit
0,West,108418.79
1,East,91522.84
2,South,46749.71
3,Central,39706.45


In [9]:
# Query 6: Top Products by Revenue
# Purpose: List top 10 products by total revenue

df6 = pd.read_sql("""
SELECT 
p.`Product Name`,
ROUND(SUM(oi.Sales),2) AS revenue
FROM order_items oi
JOIN products p
ON oi.`Product ID` = p.`Product ID`
GROUP BY p.`Product Name`
ORDER BY revenue DESC
LIMIT 10;
""", engine)

# Display result
df6

,Product Name,revenue
0,Canon imageCLASS 2200 Advanced Copier,61599.83
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.57
4,GBC DocuBind TL300 Electric Binding System,19823.48
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50
6,Hewlett Packard LaserJet 3310 Copier,18839.68
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90
8,GBC DocuBind P400 Electric Binding System,17965.07
9,High Speed Automatic Electric Letter Opener,17030.31


In [10]:
# Query 7: Category & Subcategory Sales Contribution
# Purpose: Show revenue contribution of subcategories within categories

df7 = pd.read_sql("""
WITH categorysum AS (
    SELECT 
        p.`Category` AS category,
        p.`Sub-Category` AS sub_category,
        SUM(oi.Sales) AS sub_category_revenue
    FROM products p
    JOIN order_items oi
        ON p.`Product ID` = oi.`Product ID`
    GROUP BY p.`Category`, p.`Sub-Category`
),
category_totals AS (
    SELECT
        category,
        SUM(sub_category_revenue) AS category_total
    FROM categorysum
    GROUP BY category
),
overall_total AS (
    SELECT SUM(category_total) AS overall_sales
    FROM category_totals
)
SELECT
    c.category,
    c.sub_category,
    c.sub_category_revenue,
    ROUND(c.sub_category_revenue / t.category_total * 100,2) AS subcategory_pct_of_category,
    t.category_total,
    ROUND(t.category_total / o.overall_sales * 100,2) AS category_pct_of_overall
FROM categorysum c
JOIN category_totals t
    ON c.category = t.category
CROSS JOIN overall_total o
ORDER BY category_pct_of_overall DESC;
""", engine)

df7

,category,sub_category,sub_category_revenue,subcategory_pct_of_category,category_total,category_pct_of_overall
0,Technology,Accessories,167380.31,20.02,836154.10,36.4
1,Technology,Copiers,149528.01,17.88,836154.10,36.4
2,Technology,Machines,189238.68,22.63,836154.10,36.4
3,Technology,Phones,330007.10,39.47,836154.10,36.4
4,Furniture,Bookcases,114880.05,15.48,741999.98,32.3
5,Furniture,Chairs,328449.13,44.27,741999.98,32.3
6,Furniture,Furnishings,91705.12,12.36,741999.98,32.3
7,Furniture,Tables,206965.68,27.89,741999.98,32.3
8,Office Supplies,Appliances,107532.14,14.95,719046.99,31.3
9,Office Supplies,Art,27118.80,3.77,719046.99,31.3


In [11]:
# Query 8: Discount Impact on Profit
# Purpose: Analyze how discount levels affect profit margin

df8 = pd.read_sql("""
SELECT
CASE 
    WHEN oi.Discount = 0 THEN 'No Discount'
    WHEN oi.Discount <= 0.1 THEN '0-10%'
    WHEN oi.Discount <= 0.2 THEN '10-20%'
    ELSE '20%+'
END AS discount_bucket,
ROUND(SUM(oi.Sales),2) AS total_sales,
ROUND(SUM(oi.Profit),2) AS total_profit,
ROUND(SUM(oi.Profit) / SUM(oi.Sales) * 100,2) AS profit_margin_pct
FROM order_items oi
GROUP BY discount_bucket
ORDER BY profit_margin_pct DESC;
""", engine)

df8

,discount_bucket,total_sales,total_profit,profit_margin_pct
0,No Discount,1087908.47,320987.88,29.51
1,0-10%,54369.30,9029.21,16.61
2,10-20%,792152.87,91757.14,11.58
3,20%+,362770.43,-135376.44,-37.32


In [12]:
# Query 9: Customer Lifetime Value (Top 10 customers by revenue)

df9 = pd.read_sql("""
SELECT 
c.`Customer ID`,
ROUND(SUM(oi.Sales),2) AS total_revenue
FROM customers c
JOIN orders o
    ON c.`Customer ID` = o.`Customer ID`
JOIN order_items oi
    ON o.`Order ID` = oi.`Order ID`
GROUP BY c.`Customer ID`
ORDER BY total_revenue DESC
LIMIT 10;
""", engine)

df9

,Customer ID,total_revenue
0,SM-20320,25043.07
1,TC-20980,19052.22
2,RB-19360,15117.35
3,TA-21385,14595.62
4,AB-10105,14473.57
5,KL-16645,14175.23
6,SC-20095,14142.34
7,HL-15040,12873.30
8,SE-20110,12209.44
9,CC-12370,12129.08


In [13]:
# Query 10: Market Basket Analysis
# Purpose: Identify product pairs frequently bought together

df10 = pd.read_sql("""
WITH product_pairs AS (
    SELECT 
        oi.`Order ID` AS OrderID,
        oi.`Product ID` AS Product1,
        oi2.`Product ID` AS Product2
    FROM order_items oi
    JOIN order_items oi2
        ON oi.`Order ID` = oi2.`Order ID`
        AND oi.`Product ID` < oi2.`Product ID`
)
SELECT 
    pp.Product1,
    pp.Product2,
    P1.`Product Name` AS Product1_Name,
    P2.`Product Name` AS Product2_Name,
    COUNT(*) AS times_bought_together
FROM product_pairs pp
JOIN products P1
    ON pp.Product1 = P1.`Product ID`
JOIN products P2
    ON pp.Product2 = P2.`Product ID`
GROUP BY pp.Product1, pp.Product2, P1.`Product Name`, P2.`Product Name`
HAVING COUNT(*) >= 2
ORDER BY times_bought_together DESC;
""", engine)

df10

,Product1,Product2,Product1_Name,Product2_Name,times_bought_together
0,FUR-BO-10001972,OFF-BI-10003676,O'Sullivan 4-Shelf Bookcase in Odessa Pine,"GBC Standard Recycled Report Covers, Clear Pla...",2
1,FUR-BO-10002213,OFF-LA-10002762,DMI Eclipse Executive Suite Bookcases,Avery 485,2
2,FUR-CH-10002017,OFF-AP-10001563,SAFCO Optional Arm Kit for Workspace Cribbage ...,Belkin Premiere Surge Master II 8-outlet surge...,2
3,FUR-CH-10003379,TEC-PH-10003885,Global Commerce Series High-Back Swivel/Tilt C...,Cisco SPA508G,2
4,FUR-CH-10003846,OFF-BI-10000320,Hon Valutask Swivel Chairs,GBC Plastic Binding Combs,2
5,FUR-FU-10000010,OFF-BI-10002824,"DAX Value U-Channel Document Frames, Easel Back",Recycled Easel Ring Binders,2
6,FUR-FU-10000023,OFF-BI-10000315,Eldon Wave Desk Accessories,Poly Designer Cover & Back,2
7,FUR-FU-10003464,TEC-PH-10002496,"Seth Thomas 8 1/2"" Cubicle Clock",Cisco SPA301,2
8,FUR-FU-10003832,OFF-BI-10001359,Eldon Expressions Punched Metal & Wood Desk Ac...,GBC DocuBind TL300 Electric Binding System,2
9,FUR-FU-10004973,OFF-FA-10000936,Flat Face Poster Frame,Acco Hot Clips Clips to Go,2


In [15]:
# store DataFrames from data_loading notebook for analysis_visualization

%store df1
%store df2
%store df3
%store df4
%store df5
%store df6
%store df7
%store df8
%store df9
%store df10
%store orders


print("stored successfully")

Stored 'df1' (DataFrame)
Stored 'df2' (DataFrame)
Stored 'df3' (DataFrame)
Stored 'df4' (DataFrame)
Stored 'df5' (DataFrame)
Stored 'df6' (DataFrame)
Stored 'df7' (DataFrame)
Stored 'df8' (DataFrame)
Stored 'df9' (DataFrame)
Stored 'df10' (DataFrame)
Stored 'orders' (DataFrame)
stored successfully
